# Practice data manipulation with Pandas.

In [32]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Problem 1: GroupBy Operations

Analyze batch reactor data from multiple production runs.

In [33]:
# Load batch data from file
batch_data = pd.read_csv('batch_reactor_data.csv')
batch_data.head()

,batch_id,reactor,shift,temperature,pressure,feed_rate,conversion,yield,impurity
0,B0001,R3,night,436.6,1.89,145.0,0.775,0.650,0.089
1,B0002,R2,night,431.7,1.92,128.6,0.816,0.880,0.073
2,B0003,R2,night,375.8,3.69,58.9,0.846,0.793,0.087
3,B0004,R3,day,367.1,1.08,91.8,0.796,0.739,0.075
4,B0005,R3,night,416.9,1.42,137.9,0.603,0.562,0.150


**1a.** Calculate the mean yield and impurity for each reactor.

In [34]:
batch_data.groupby('reactor')[['yield', 'impurity']].mean() #only use 2 list of columns

,yield,impurity
reactor,,
R1,0.691792,0.096506
R2,0.697439,0.097379
R3,0.691561,0.096386


**1b.** Calculate the mean yield grouped by both reactor AND shift. Which combination performs best?

In [35]:
mean_yield = batch_data.groupby(['reactor', 'shift'])['yield'].mean()
print(mean_yield)

mean_yield.idxmax() # to find the best combination


reactor  shift
R1       day      0.682595
         night    0.702829
R2       day      0.718531
         night    0.677588
R3       day      0.667536
         night    0.714759
Name: yield, dtype: float64


('R2', 'day')

So, the combination of R2+day performs best.


**1c.** Use `.agg()` to calculate multiple statistics (mean, std, min, max) for yield by reactor.

In [36]:
batch_data.groupby('reactor')['yield'].agg(['mean', 'std', 'min', 'max'])
#4 values but 2 list of columns


,mean,std,min,max
reactor,,,,
R1,0.691792,0.116302,0.502,0.895
R2,0.697439,0.118531,0.503,0.888
R3,0.691561,0.121434,0.507,0.876


## Problem 2: Merging DataFrames

Combine data from different sources.

In [37]:
# Load reactor specifications data
reactor_specs = pd.read_csv('reactor_specs.csv')

# Quality control results (small dataset for merge demonstration)
qc_results = pd.DataFrame({
    'batch_id': ['B0001', 'B0002', 'B0003', 'B0005', 'B0010'],
    'passed_qc': [True, True, False, True, True],
    'qc_notes': ['OK', 'OK', 'Color off-spec', 'OK', 'OK']
})

# print("Reactor specs:")
# print(reactor_specs)
# print("\nQC results:")
# print(qc_results)

display(reactor_specs)
display(qc_results)

,reactor,volume_L,max_temp,age_years
0,R1,1000,500,8
1,R2,1500,480,7
2,R3,800,520,6


,batch_id,passed_qc,qc_notes
0,B0001,True,OK
1,B0002,True,OK
2,B0003,False,Color off-spec
3,B0005,True,OK
4,B0010,True,OK


**2a.** Merge batch_data with reactor_specs to add reactor volume to each batch. How many batches don't have matching reactor specs?

In [38]:
merged_data = pd.merge(batch_data, reactor_specs, on='reactor', how='left')
merged_data.head()

,batch_id,reactor,shift,temperature,pressure,feed_rate,conversion,yield,impurity,volume_L,max_temp,age_years
0,B0001,R3,night,436.6,1.89,145.0,0.775,0.650,0.089,800,520,6
1,B0002,R2,night,431.7,1.92,128.6,0.816,0.880,0.073,1500,480,7
2,B0003,R2,night,375.8,3.69,58.9,0.846,0.793,0.087,1500,480,7
3,B0004,R3,day,367.1,1.08,91.8,0.796,0.739,0.075,800,520,6
4,B0005,R3,night,416.9,1.42,137.9,0.603,0.562,0.150,800,520,6


In [39]:
merged_data['volume_L'].isna().sum()

np.int64(0)

All matched.


**2b.** Merge batch_data with qc_results using a left join. Fill missing QC values appropriately.

In [40]:
merged_qc = pd.merge(batch_data, qc_results, on='batch_id', how='left')
#combine together

merged_qc['passed_qc'] = (
    merged_qc['passed_qc']
    .astype('boolean')
    .fillna(False)
)
#for the a few times, i got a warning message, so I added the .astype('boolean') to avoid the warning message

merged_qc['qc_notes'] = merged_qc['qc_notes'].fillna('Not tested')
#fill in the missing values

merged_qc.head() #display the first 5 rows

,batch_id,reactor,shift,temperature,pressure,feed_rate,conversion,yield,impurity,passed_qc,qc_notes
0,B0001,R3,night,436.6,1.89,145.0,0.775,0.650,0.089,True,OK
1,B0002,R2,night,431.7,1.92,128.6,0.816,0.880,0.073,True,OK
2,B0003,R2,night,375.8,3.69,58.9,0.846,0.793,0.087,False,Color off-spec
3,B0004,R3,day,367.1,1.08,91.8,0.796,0.739,0.075,False,Not tested
4,B0005,R3,night,416.9,1.42,137.9,0.603,0.562,0.150,True,OK


**2c.** Is there a relationship between reactor age (age_years) and yield? Calculate the correlation.

In [41]:
merged_data['age_years'].corr(merged_data['yield'])

np.float64(-0.0011926905654343696)

The correlation is approximately -0.0012 near 0, indicating no linear relationship between reactor age and yield.

## Problem 3: Pivot Tables and Reshaping

**3a.** Create a pivot table showing mean yield for each reactor (rows) and shift (columns).

In [42]:
batch_data.pivot_table(
    values='yield',
    index='reactor',
    columns='shift',
    aggfunc='mean'
)

shift,day,night
reactor,,
R1,0.682595,0.702829
R2,0.718531,0.677588
R3,0.667536,0.714759


**3b.** Create temperature bins (340-345, 345-350, 350-355, 355-360) and analyze yield by temperature bin.

In [43]:
batch_data['temperature'].min(), batch_data['temperature'].max() #check the range first

(np.float64(350.5), np.float64(449.7))

In [44]:
batch_data['temp_bin'] = pd.cut(
    batch_data['temperature'],
    bins=[340, 345, 350, 355, 360]
)

batch_data.groupby('temp_bin', observed=True)['yield'].mean()
#from the checking range, we know we don't have the 340-345 and 345-350 range

temp_bin
(350, 355]    0.719714
(355, 360]    0.700286
Name: yield, dtype: float64

**3c.** Based on your analysis, what would you recommend to improve reaction yield

Based on the analysis, I would recommend operating at a temperature between 350–355°C because this range has the highest mean yield. And R1 may be operated during the night shift, while R2 should preferably be operated during the day shift.